# Desafío por equipos · Estadística y Python aplicado 

**Caso:** Mercado Sur · **Pregunta de la clase:** ¿la experiencia del canal web empeoró, o cambió la composición de clientes y operaciones observadas?

Son **siete equipos y siete pedidos distintos**: cada equipo mira la misma pregunta desde un ángulo. Al final, los siete pedidos juntos arman la respuesta.

Trabajen solo con estos datos, que son ficticios y reproducibles.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = False          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## Preparación

In [ ]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas", "matplotlib")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

VIOLETA, VERDE, NARANJA, GRIS = "#7C5CF6", "#10D9A0", "#F59E0B", "#8888A8"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11, "axes.titleweight": "bold"})
pd.options.display.float_format = "{:,.2f}".format

# --- los datos de Mercado Sur, generados con código (misma semilla de toda la cursada) ---
SEMILLA = 2025                                 # el «número de sorteo»
rng = np.random.default_rng(SEMILLA)           # el generador de sorteos
N_CLIENTES = 1500
N_OPS = 12000                                  # operaciones que se sortean (algunas se descartan más adelante)
print("Listo: vamos a generar", N_CLIENTES, "clientes y", N_OPS, "operaciones")
CATEGORIAS = {"Electrónica": (150, 900, 0.15), "Hogar": (30, 300, 0.20), "Indumentaria": (20, 150, 0.20),
              "Deportes": (25, 250, 0.15), "Alimentos": (3, 40, 0.30)}   # (precio mínimo, precio máximo, peso en las ventas)
filas, n = [], 1
for cat, (lo, hi, _) in CATEGORIAS.items():
    for _ in range(8):
        filas.append({"id_producto": f"P{n:03d}", "categoria": cat, "nombre": f"{cat} {n:02d}",
                      "precio_lista": round(float(rng.uniform(lo, hi)), 2)})
        n += 1
productos = pd.DataFrame(filas)
productos.head()
REGIONES = ["Centro", "Norte", "Sur", "Litoral", "Cuyo"]
clientes = pd.DataFrame({
    "id_cliente": [f"C{i:04d}" for i in range(1, N_CLIENTES + 1)],
    "region": rng.choice(REGIONES, size=N_CLIENTES, p=[0.30, 0.20, 0.18, 0.17, 0.15]),
    "fecha_alta": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, N_CLIENTES), unit="D"),
})
clientes["fecha_alta"] = clientes["fecha_alta"].dt.strftime("%Y-%m-%d")
clientes.head()
fechas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, N_OPS), unit="D")
ventas = pd.DataFrame({
    "id_venta": [f"V{i:06d}" for i in range(1, N_OPS + 1)],
    "fecha": fechas,
    "id_cliente": rng.choice(clientes["id_cliente"], size=N_OPS),
    "canal": rng.choice(["web", "tienda"], size=N_OPS, p=[0.55, 0.45]),
})
pesos = np.array([v[2] for v in CATEGORIAS.values()])
cats = np.array(list(CATEGORIAS.keys()))[rng.choice(len(CATEGORIAS), size=N_OPS, p=pesos / pesos.sum())]
ventas["id_producto"] = [rng.choice(productos.loc[productos["categoria"] == c, "id_producto"]) for c in cats]
ventas = ventas.merge(productos[["id_producto", "categoria", "precio_lista"]], on="id_producto", how="left")
ventas = ventas.merge(clientes[["id_cliente", "region"]], on="id_cliente", how="left")
ventas["unidades"] = np.where(ventas["categoria"] == "Alimentos", rng.integers(1, 9, N_OPS), rng.integers(1, 5, N_OPS))
ventas["precio_unitario"] = (ventas["precio_lista"] * rng.uniform(0.97, 1.05, N_OPS)).round(2)
ventas["descuento_pct"] = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], size=N_OPS, p=[0.45, 0.15, 0.20, 0.12, 0.08])
print(f"{len(ventas):,} operaciones sorteadas")
s2 = ventas["fecha"].dt.month > 6
seg = (ventas["canal"] == "web") & (ventas["region"] == "Litoral") & (ventas["categoria"] == "Electrónica") & s2
seg_indum = (ventas["canal"] == "web") & (ventas["categoria"] == "Indumentaria") & s2
p_comp, p_dev, p_can = np.full(N_OPS, 0.88), np.full(N_OPS, 0.07), np.full(N_OPS, 0.05)
p_comp[seg], p_dev[seg], p_can[seg] = 0.64, 0.30, 0.06
p_comp[seg_indum], p_dev[seg_indum], p_can[seg_indum] = 0.81, 0.13, 0.06
u = rng.random(N_OPS)
ventas["estado"] = np.where(u < p_comp, "completada", np.where(u < p_comp + p_dev, "devuelta", "cancelada"))
ventas = ventas.loc[~(seg & (rng.random(N_OPS) < 0.38))].copy()
print(f"Estados asignados · operaciones: {len(ventas):,}")
n_v = len(ventas)
sat = rng.choice([1, 2, 3, 4, 5], size=n_v, p=[0.05, 0.08, 0.17, 0.35, 0.35]).astype(float)
dev = (ventas["estado"] == "devuelta").to_numpy()
can = (ventas["estado"] == "cancelada").to_numpy()
sat[dev] = rng.choice([1, 2, 3], size=dev.sum(), p=[0.5, 0.3, 0.2])               # las devueltas califican peor
sat[rng.random(n_v) < np.where(can, 0.80, np.where(dev, 0.25, 0.12))] = np.nan     # y los faltantes
ventas["satisfaccion"] = sat

ventas = ventas.sort_values("fecha").reset_index(drop=True)                          # ordenadas por fecha
ventas["id_venta"] = [f"V{i:06d}" for i in range(1, len(ventas) + 1)]
ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
ventas = ventas[["id_venta", "fecha", "id_cliente", "id_producto", "canal", "unidades",
                 "precio_unitario", "descuento_pct", "estado", "satisfaccion"]]       # descartamos las columnas auxiliares
ventas.head()

# --- una sola tabla lista para analizar ---
ventas["fecha"] = pd.to_datetime(ventas["fecha"])
ventas["neto"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento_pct"])
ventas["semestre"] = np.where(ventas["fecha"].dt.month <= 6, "S1", "S2")
base = (ventas.merge(productos[["id_producto", "categoria"]], on="id_producto", how="left")
              .merge(clientes[["id_cliente", "region"]], on="id_cliente", how="left"))
assert len(base) == len(ventas), "¡El merge duplicó o perdió filas!"

def cinco_controles(v, c, p):
    """La herramienta de la clase: cinco controles de calidad sobre las tablas."""
    controles = {
        "1 · Grano: id_venta es único (una fila = una operación)": v["id_venta"].is_unique,
        "2 · Claves: todas las ventas tienen cliente y producto": bool(v["id_cliente"].isin(c["id_cliente"]).all() and v["id_producto"].isin(p["id_producto"]).all()),
        "3 · Fechas: todas dentro de 2025": bool(v["fecha"].between("2025-01-01", "2025-12-31").all()),
        "4 · Estados: solo completada, devuelta o cancelada": bool(v["estado"].isin(["completada", "devuelta", "cancelada"]).all()),
        "5 · Valores: descuento entre 0 y 1 y unidades positivas": bool(v["descuento_pct"].between(0, 1).all() and (v["unidades"] > 0).all()),
    }
    for descripcion, ok in controles.items():
        print("✔" if ok else "✘", descripcion)
    return all(controles.values())

print(f"Listo · tabla `base`: {len(base):,} operaciones · columnas: {', '.join(base.columns)}")

## Tu equipo y tu pedido

Cambiá `MI_EQUIPO` por el número de tu equipo (1 a 7) y corré la celda. Te muestra la pregunta, el universo, lo que tienen que entregar y el error a evitar.

In [ ]:
PEDIDOS = {
    1: dict(titulo='El ticket del canal web', pregunta='¿Cambió el ticket del canal web entre el primer y el segundo semestre?', universo='Operaciones web completadas. El ticket de una operación es su `neto`.', entregar=['Una tabla con la cantidad de operaciones, la media y la mediana de cada semestre, y la variación porcentual', 'Un gráfico que compare la distribución del ticket entre semestres', 'Un hallazgo que separe evidencia, interpretación y límite'], cuidado='Usar solo la media sin mirar la mediana, y no reportar cuántas operaciones hay en cada semestre.'),
    2: dict(titulo='La recompra en la web', pregunta='¿Cambió la proporción de compradores recurrentes en la web entre semestres?', universo='Clientes con al menos una operación web completada en el semestre. Es recurrente quien tiene al menos dos operaciones web completadas en ese mismo semestre.', entregar=['Una tabla con compradores, recurrentes y porcentaje de recurrentes de cada semestre', 'Un gráfico de barras que compare los dos semestres', 'Un hallazgo que separe evidencia, interpretación y límite'], cuidado='Contar operaciones en lugar de clientes, o cambiar el universo de un semestre al otro.'),
    3: dict(titulo='La satisfacción en la web', pregunta='¿Cambió la satisfacción de las operaciones web entre semestres?', universo='Operaciones web de cualquier estado. Los faltantes se analizan y no se reemplazan.', entregar=['Una tabla con respuestas, porcentaje de faltantes, media y mediana de cada semestre', 'El porcentaje de faltantes por estado en cada semestre', 'Un gráfico y un hallazgo que separe evidencia, interpretación y límite'], cuidado='Reemplazar los faltantes por 0 o por la media, y no decir qué universo se usó.'),
    4: dict(titulo='Los estados de las operaciones web', pregunta='¿Cambió la mezcla de estados de las operaciones web y dónde se concentra el cambio?', universo='Operaciones web de cualquier estado.', entregar=['El porcentaje de completadas, devueltas y canceladas de cada semestre', 'La tasa de devolución por categoría y por región, con la cantidad de operaciones de cada grupo', 'Un gráfico y un hallazgo que diga dónde se concentra el cambio y qué no se puede afirmar'], cuidado='Afirmar un cambio en un grupo sin mirar cuántas operaciones tiene.'),
    5: dict(titulo='La composición de las operaciones web', pregunta='¿Cambió la composición de las operaciones web por región y por categoría?', universo='Operaciones web de cualquier estado.', entregar=['Una tabla con el porcentaje de operaciones por región y por categoría en cada semestre, y la variación en puntos porcentuales', 'Un gráfico que compare la composición entre semestres', 'Un hallazgo que diga qué cambió en la mezcla y qué no se puede afirmar'], cuidado='Confundir la variación en puntos porcentuales con la variación porcentual.'),
    6: dict(titulo='La frecuencia de compra en la web', pregunta='¿Cambió la frecuencia de compra de los clientes del canal web entre semestres?', universo='Clientes con al menos una operación web completada en el semestre. La frecuencia es la cantidad de operaciones web completadas del cliente en ese semestre.', entregar=['Una tabla con compradores, media y mediana de operaciones por comprador de cada semestre', 'La distribución de compradores según su cantidad de operaciones (1, 2, 3, 4, 5 o más)', 'Un gráfico y un hallazgo que separe evidencia, interpretación y límite'], cuidado='Promediar sin mirar la distribución, o incluir operaciones devueltas y canceladas.'),
    7: dict(titulo='La tienda como grupo de comparación', pregunta='¿Lo que cambió en la web también cambió en la tienda?', universo='Operaciones de web y de tienda, de cualquier estado. La tienda es el grupo de comparación: se mide igual que la web.', entregar=['Una tabla con operaciones, tasa de devolución, ticket mediano y satisfacción media de web y tienda en cada semestre', 'La tasa de devolución de Indumentaria en web y en tienda, con la cantidad de operaciones', 'Un gráfico y un hallazgo que diga qué es propio de la web y qué no se puede afirmar'], cuidado='Medir la tienda con otro universo que la web, o decir que algo es propio de la web sin haberlo mirado en la tienda.'),
}

MI_EQUIPO = 1          # <- el número de tu equipo, del 1 al 7

p = PEDIDOS[MI_EQUIPO]
print(f"EQUIPO {MI_EQUIPO} · {p['titulo']}\n")
print("Pregunta: ", p["pregunta"])
print("Universo: ", p["universo"])
print("\nQué entregar:")
for e in p["entregar"]: print("  •", e)
print("\nCuidado con:", p["cuidado"])
print("\nPrimer semestre (S1): enero a junio · Segundo semestre (S2): julio a diciembre")

## Reglas comunes de Mercado Sur

- **Facturación y ticket:** solo operaciones `completada`, salvo que el pedido diga otro universo.
- **Recurrente:** al menos dos operaciones completadas en el período analizado.
- **Satisfacción:** puede faltar. Los faltantes se **analizan y no se reemplazan**.
- **Cada número declara:** fórmula, universo, período y tratamiento de estados, y la cantidad (`n`) en que se apoya.
- Ya hay una tabla `base` con todo lo necesario: `semestre`, `categoria`, `region`, `neto` y los estados.

## Tu turno

### Paso 1 · Definí el universo y la medida

Escriban, en una celda de texto, cómo definen lo que van a medir: **universo, período, grano y fórmula**.

In [ ]:
# 👉 TU TURNO · Paso 1: filtrá el universo de tu pedido (por ejemplo, el canal web)

### Paso 2 · Calculá la tabla de tu pedido

In [ ]:
# 👉 TU TURNO · Paso 2: la tabla comparando S1 y S2, con la cantidad (n) de cada grupo

### Paso 3 · La visualización

Un gráfico que se entienda sin explicación: con título que diga qué muestra y ejes con nombre.

In [ ]:
# 👉 TU TURNO · Paso 3: el gráfico de tu pedido

### Paso 4 · Los cinco controles

La función `cinco_controles` de la clase verifica las tablas. Corranla, y agreguen al menos un control propio de **su** indicador (por ejemplo, que el universo no esté vacío o que los porcentajes sumen 100).

In [ ]:
# 👉 TU TURNO · Paso 4
cinco_controles(ventas, clientes, productos)

### Paso 5 · Hallazgo y límite

Separen **tres cosas**: la **evidencia** (el número), la **interpretación** (qué creen que significa) y el **límite** (qué no pueden afirmar).

---

**Ficha de evidencia · Equipo: ________ · Pedido: ________**

- **Pregunta:**
- **Definición** (universo, período, grano y fórmula):
- **Procedimiento** (qué hicieron, en orden):
- **Resultado** (los números que sostienen el hallazgo):
- **Límite** (qué no pueden afirmar):